# Calc Flow DataFusion quickstart

Calculate gross order totals from an Arrow table, then filter them in a SQL stage.
Install the published Calc Flow release in the environment used by your Jupyter kernel.

```bash
python -m pip install "calc-flow-python==2026.11.29"
```

Jupyter's IPython kernel runs an event loop, so the execution cells use top-level
`await`. The [standalone Python example](../01_datafusion_pipeline.py) uses
synchronous `cf.compute()` and `Program.collect()`.

In [ ]:
from __future__ import annotations

import pyarrow as pa

import calc_flow as cf

orders = pa.table(
    {
        "order_id": ["A-100", "A-101", "A-102"],
        "quantity": [3, 1, 4],
        "unit_price": [10, 12, 10],
    }
)

## Calculate and filter order totals

Declare the new column with `with_columns`. The SQL stage reads the previous stage
as `input`. `compute_async` supplies the Arrow schema and returns an Arrow table.

In [ ]:
def large_orders(source: cf.TableExpr) -> cf.TableExpr:
    return source.with_columns(gross=source["quantity"] * source["unit_price"]).sql(
        "SELECT order_id, gross FROM input WHERE gross >= 20 ORDER BY order_id"
    )


expected = {"order_id": ["A-100", "A-102"], "gross": [30, 40]}
result = await cf.compute_async(orders, large_orders)
if result.to_pydict() != expected:
    raise RuntimeError(f"Unexpected order totals: {result.to_pydict()}")
result

## Reuse the declaration in a batch program

A named `Program` keeps the declaration separate from the input table and returns
each output under its logical name.

In [ ]:
source = cf.table_input("orders", schema=orders.schema)
program = cf.Program(
    "order-totals", mode="batch", outputs={"totals": large_orders(source)}
)
outputs = await program.collect_async({"orders": orders})
if outputs["totals"].to_pydict() != expected:
    raise RuntimeError(f"Unexpected program totals: {outputs['totals'].to_pydict()}")
outputs["totals"]